# Carpet WaveToy Thorns

Generate Carpet-style WaveToy thorns and inspect the CCL and source files they provide.

Navigation: [Index](../index.ipynb) | Previous: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb) | Next: [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)


## Learning Goals

- Generate a small set of Einstein Toolkit thorns for the wave equation.
- Separate evolution, initial-data, and diagnostic roles.
- Read schedule entries as the order of operations in a Toolkit run.

## Words for This Notebook

- **Carpet:** an Einstein Toolkit mesh driver used by many older Toolkit examples.
- **Evolution thorn:** the component that updates fields in time.
- **Initial-data thorn:** the component that sets starting field values.
- **Diagnostic thorn:** the component that writes numbers used to check the run.

Use the code cells actively: first predict what should happen, then run the cell, then explain the output in plain language. This predict-run-explain pattern keeps the physics idea connected to the programming details.


## Generate Carpet WaveToy Thorns
Compiling or running these thorns requires an external Einstein Toolkit checkout. This notebook focuses on generation and inspection.

## Import Thorn Inspection Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.

If you are new to Python, do not study this helper cell line by line on a first pass. Its job is practical: run terminal commands, shorten long command output, and stop with a clear message if a required tool is missing. Focus first on the cells that generate, inspect, build, run, and interpret the physics project.


In [1]:
from pathlib import Path
import re, shutil, subprocess, sys, tempfile


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)


def require_toolchain():
    if shutil.which("make") is None:
        raise RuntimeError(
            "This notebook requires make to build the generated project."
        )
    if not any(shutil.which(name) for name in ["cc", "gcc", "clang"]):
        raise RuntimeError(
            "This notebook requires a C compiler such as cc, gcc, or clang."
        )


## Create a Carpet Thorn Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [2]:
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_carpet_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / "et_wavetoy"


## Generate Carpet-Compatible Thorns

This command invokes the same module a learner can run from a terminal and then verifies that the project directory exists.


In [3]:
command = [sys.executable, "-m", "nrpy.examples.carpet_wavetoy_thorns"]
print("generator command: python -m nrpy.examples.carpet_wavetoy_thorns")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)


generator command: python -m nrpy.examples.carpet_wavetoy_thorns


In 0.063s, worker completed task 'register_CFunction_rhs_eval'
In 0.094s, worker completed task 'register_CFunction_exact_solution_all_points'
In 0.095s, worker completed task 'register_CFunction_exact_solution_all_points'
Checking <workspace>/project/et_wavetoy/WaveToyNRPy/schedule.ccl...[written]
Checking <workspace>/project/et_wavetoy/WaveToyNRPy/interface.ccl...[written]
Checking <workspace>/project/et_wavetoy/WaveToyNRPy/param.ccl...[written]
Checking <workspace>/project/et_wavetoy/IDWaveToyNRPy/schedule.ccl...[written]
Checking <workspace>/project/et_wavetoy/IDWaveToyNRPy/interface.ccl...[written]
Checking <workspace>/project/et_wavetoy/IDWaveToyNRPy/param.ccl...[written]
Checking <workspace>/project/et_wavetoy/diagWaveToyNRPy/schedule.ccl...[written]
Checking <workspace>/project/et_wavetoy/diagWaveToyNRPy/interface.ccl...[written]
Checking <workspace>/project/et_wavetoy/diagWaveToyNRPy/param.ccl...[written]
Checking <workspace>/project/et_wavetoy/WaveToyNRPy/src/WaveToyNRPy_MoL_

## Step 4: Inspect the generated inventory

The inventory identifies the generated files relevant to this lesson.

In [4]:
print("thorn inventory:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))


thorn inventory:
IDWaveToyNRPy/interface.ccl
IDWaveToyNRPy/param.ccl
IDWaveToyNRPy/schedule.ccl
IDWaveToyNRPy/src/IDWaveToyNRPy_exact_solution_all_points.c
IDWaveToyNRPy/src/make.code.defn
WaveToyNRPy/interface.ccl
WaveToyNRPy/param.ccl
WaveToyNRPy/schedule.ccl
WaveToyNRPy/src/WaveToyNRPy_MoL_registration.c
WaveToyNRPy/src/WaveToyNRPy_Symmetry_registration_oldCartGrid3D.c
WaveToyNRPy/src/WaveToyNRPy_rhs_eval.c
WaveToyNRPy/src/WaveToyNRPy_specify_Driver_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_specify_NewRad_BoundaryConditions_parameters.c
WaveToyNRPy/src/WaveToyNRPy_specify_aux_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_specify_evol_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_zero_rhss.c
WaveToyNRPy/src/make.code.defn
WaveToyNRPy/src/simd/simd_intrinsics.h
diagWaveToyNRPy/interface.ccl
diagWaveToyNRPy/param.ccl
diagWaveToyNRPy/schedule.ccl
diagWaveToyNRPy/src/diagWaveToyNRPy_exact_solution_all_points.c
diagWaveToyNRPy/src/make.code.defn


## Step 5: Inspect the evolution thorn variable declarations

The `interface.ccl` file declares the evolved variables and inherited Einstein Toolkit capabilities.


In [5]:
relative_path = "WaveToyNRPy/interface.ccl"
print(f"--- {relative_path} ---")
print((PROJECT_DIR / relative_path).read_text(encoding="utf-8", errors="replace"))


--- WaveToyNRPy/interface.ccl ---

# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: WaveToyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Boundary Grid MethodofLines

# Needed functions and #include's:
USES INCLUDE: Symmetry.h
USES INCLUDE: Boundary.h


# Needed Method of Lines function
CCTK_INT FUNCTION MoLRegisterEvolvedGroup(CCTK_INT IN EvolvedIndex, CCTK_INT IN RHSIndex)
REQUIRES FUNCTION MoLRegisterEvolvedGroup

# Needed Boundary Conditions function
CCTK_INT FUNCTION GetBoundarySpecification(CCTK_INT IN size, CCTK_INT OUT ARRAY nboundaryzones, CCTK_INT OUT ARRAY is_internal, CCTK_INT OUT ARRAY is_staggered, CCTK_INT OUT ARRAY shiftout)
USES FUNCTION GetBoundarySpecification

CCTK_INT FUNCTION S

## Step 6: List the evolution thorn schedule entries

The schedule entries show where the right-hand-side and boundary hooks enter the Toolkit time step.


In [6]:
relative_path = "WaveToyNRPy/schedule.ccl"
print(f"--- {relative_path}: schedule entries ---")
for line in (
    (PROJECT_DIR / relative_path)
    .read_text(encoding="utf-8", errors="replace")
    .splitlines()
):
    if line.startswith("schedule "):
        print(line)


--- WaveToyNRPy/schedule.ccl: schedule entries ---
schedule WaveToyNRPy_specify_Driver_BoundaryConditions in Driver_BoundarySelect
schedule WaveToyNRPy_Symmetry_registration_oldCartGrid3D at BASEGRID as Symmetry_registration
schedule WaveToyNRPy_zero_rhss at BASEGRID after Symmetry_registration
schedule WaveToyNRPy_MoL_registration in MoL_Register
schedule WaveToyNRPy_rhs_eval in MoL_CalcRHS as rhs_eval
schedule WaveToyNRPy_specify_NewRad_BoundaryConditions_parameters in MoL_CalcRHS after WaveToyNRPy_RHS
schedule WaveToyNRPy_specify_evol_BoundaryConditions in MoL_PostStep
schedule GROUP ApplyBCs as WaveToyNRPy_evol_ApplyBCs in MoL_PostStep after WaveToyNRPy_specify_evol_BoundaryConditions
schedule WaveToyNRPy_specify_aux_BoundaryConditions in MoL_PseudoEvolution after WaveToyNRPy_BSSN_constraints
schedule GROUP ApplyBCs as WaveToyNRPy_aux_ApplyBCs in MoL_PseudoEvolution after WaveToyNRPy_specify_aux_BoundaryConditions


## Step 7: Inspect the initial-data thorn variable declarations

The initial-data interface declares the fields initialized before time evolution begins.


In [7]:
relative_path = "IDWaveToyNRPy/interface.ccl"
print(f"--- {relative_path} ---")
print((PROJECT_DIR / relative_path).read_text(encoding="utf-8", errors="replace"))


--- IDWaveToyNRPy/interface.ccl ---

# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: IDWaveToyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Grid WaveToyNRPy # WaveToyNRPy provides all gridfunctions.

# Needed functions and #include's:


# FIXME: add info for symmetry conditions:
#    https://einsteintoolkit.org/thornguide/CactusBase/SymBase/documentation.html

# Tell the Toolkit that we want all gridfunctions
#    to be visible to other thorns by using
#    the keyword "public". Note that declaring these
#    gridfunctions *does not* allocate memory for them;
#    that is done by the schedule.ccl file.

public:



## Step 8: Inspect the diagnostics thorn schedule

The diagnostics schedule shows where output is written during the run.

In [8]:
relative_path = "WaveToyNRPy/schedule.ccl"
print(f"--- {relative_path}: schedule entries ---")
for line in (
    (PROJECT_DIR / relative_path)
    .read_text(encoding="utf-8", errors="replace")
    .splitlines()
):
    if line.startswith("schedule "):
        print(line)


--- WaveToyNRPy/schedule.ccl: schedule entries ---
schedule WaveToyNRPy_specify_Driver_BoundaryConditions in Driver_BoundarySelect
schedule WaveToyNRPy_Symmetry_registration_oldCartGrid3D at BASEGRID as Symmetry_registration
schedule WaveToyNRPy_zero_rhss at BASEGRID after Symmetry_registration
schedule WaveToyNRPy_MoL_registration in MoL_Register
schedule WaveToyNRPy_rhs_eval in MoL_CalcRHS as rhs_eval
schedule WaveToyNRPy_specify_NewRad_BoundaryConditions_parameters in MoL_CalcRHS after WaveToyNRPy_RHS
schedule WaveToyNRPy_specify_evol_BoundaryConditions in MoL_PostStep
schedule GROUP ApplyBCs as WaveToyNRPy_evol_ApplyBCs in MoL_PostStep after WaveToyNRPy_specify_evol_BoundaryConditions
schedule WaveToyNRPy_specify_aux_BoundaryConditions in MoL_PseudoEvolution after WaveToyNRPy_BSSN_constraints
schedule GROUP ApplyBCs as WaveToyNRPy_aux_ApplyBCs in MoL_PseudoEvolution after WaveToyNRPy_specify_aux_BoundaryConditions


## Step 9: Inspect One Generated Source File

One right-hand-side source file represents the generated C output path.


In [9]:
source_path = PROJECT_DIR / "WaveToyNRPy" / "src" / "WaveToyNRPy_rhs_eval.c"
if not source_path.is_file():
    raise FileNotFoundError(source_path)
print(source_path.read_text(encoding="utf-8", errors="replace"))


#include "cctk.h"
#include "cctk_Arguments.h"
#include "cctk_Parameters.h"
#include "math.h"
#include "simd/simd_intrinsics.h"

/**
 * Set RHSs for wave equation.
 */
void WaveToyNRPy_rhs_eval(CCTK_ARGUMENTS) {
  DECLARE_CCTK_ARGUMENTS_WaveToyNRPy_rhs_eval;
  const CCTK_REAL *restrict NOSIMDwavespeed = CCTK_ParameterGet("wavespeed", "IDWaveToyNRPy", NULL); // IDWaveToyNRPy::wavespeed
  const REAL_SIMD_ARRAY wavespeed = ConstSIMD(*NOSIMDwavespeed);                                     // IDWaveToyNRPy::wavespeed
  const CCTK_REAL NOSIMDinvdxx0 = 1.0 / CCTK_DELTA_SPACE(0);
  const REAL_SIMD_ARRAY invdxx0 = ConstSIMD(NOSIMDinvdxx0);
  const CCTK_REAL NOSIMDinvdxx1 = 1.0 / CCTK_DELTA_SPACE(1);
  const REAL_SIMD_ARRAY invdxx1 = ConstSIMD(NOSIMDinvdxx1);
  const CCTK_REAL NOSIMDinvdxx2 = 1.0 / CCTK_DELTA_SPACE(2);
  const REAL_SIMD_ARRAY invdxx2 = ConstSIMD(NOSIMDinvdxx2);
#pragma omp parallel for
  for (int i2 = cctk_nghostzones[2]; i2 < cctk_lsh[2] - cctk_nghostzones[2]; i2++) {
    for (in

The thorn inventory, CCL files, and full generated source file show the three pieces of the Carpet WaveToy example: evolution, initial data, and diagnostics. Read the source by looking first for the scheduled function name, then the grid loop, then the right-hand-side assignment.


## Learning Check

Before inspecting the inventory, predict why the project needs more than one thorn. After inspection, assign each thorn a role in one sentence.


## Continue to Code-Writing Choices
- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)
- [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)
